### Python 的异步编程

```
异步编程（尤其是 asyncio + httpx）是当前大模型应用开发（如 OpenAI、LangChain、Coze 等 SDK）的底层标配。Python 的异步并不是多线程/多进程，而是单线程并发，专门用来解决 I/O 密集型任务的“等待浪费”问题（比如等大模型返回结果、等数据库查询、等网络响应）

## 1. async/await 语法基础（核心认知）

* async def：定义一个协程函数，调用它不会立即执行，而是返回一个协程对象。

* await：挂起当前协程，让出 CPU 控制权，等待耗时的 I/O 操作完成。注意：await 只能在 async def 内部使用。

* 禁忌：⚠️ 在 async 函数中，绝对不要用 time.sleep()，因为它会阻塞整个线程。必须用 await asyncio.sleep()。

* asyncio.create_task:创建异步协程任务并立即执行，但不返回结果，通过await获取结果

* asyncio.gather:并发启动多个异步协程任务，并统一返回结果（结果一般通过list处理成列表）

In [ ]:
import asyncio
import time

async def say_after(delay, word):
    print(f"开始等待 {delay}s...")
    await asyncio.sleep(delay)  # 模拟 I/O，如网络请求
    print(word)
    return f"完成: {word}"

# 方式1：直接运行协程（必须用 asyncio.run）
async def main():
    # 注意：如果直接 await 调用，它们是串行执行的
    start = time.perf_counter()

    # await say_after(2, "Hello")
    # await say_after(1, "World")
    # 总耗时：2 + 1 = 3秒

    # 并行执行
    tasks = [
       say_after(2, "Hello"),
       say_after(1, "Hello")
    ]
    await asyncio.gather(*tasks)
    diff = time.perf_counter() - start
    print(f"耗时{diff}")
asyncio.run(main())

## 2. asyncio.gather() —— 并发执行（最常用）

* gather 能同时启动多个异步任务，并等待所有任务完成后，按原顺序返回结果

* 实际场景：同时向 3 个大模型（GPT、Claude、Gemini）发送请求，取最快返回的结果

In [ ]:
async def main():
    # 同时启动 3 个任务，总耗时约 2 秒（而不是 2+1+1.5=4.5 秒）
    results = await asyncio.gather(
        say_after(2, "任务A"),
        say_after(1, "任务B"),
        say_after(1.5, "任务C")
    )
    print(f"全部结果: {results}")

asyncio.run(main())
# 输出顺序（注意时间戳）：
# 开始等待 2s...
# 开始等待 1s...
# 开始等待 1.5s...
# World (约1s后)
# 完成: 任务C (约1.5s后)
# Hello (约2s后)
# 全部结果: ['完成: 任务A', '完成: 任务B', '完成: 任务C']

## 3. asyncio.create_task() —— 任务管理（更精细）

* create_task 与 gather 的区别在于：create_task 是“下发”任务（立即在后台调度），但你需要自己负责收集结果（用 await task）。适合需要单独控制或取消某个任务的场景。

In [ ]:
async def main():
    # 创建任务（立即开始运行，不会阻塞）
    task1 = asyncio.create_task(say_after(2, "任务1"))
    task2 = asyncio.create_task(say_after(1, "任务2"))

    print("任务已创建，主协程继续执行...")
    # 这里还可以做一些其他非异步的计算
    await asyncio.sleep(0.5)

    # 等待任务完成并获取结果
    result1 = await task1
    result2 = await task2
    print(f"结果: {result1}, {result2}")

asyncio.run(main())

## 4. httpx —— 异步 HTTP 客户端（替代 requests）

* 这是现代 Python 异步 web 请求的标准方案（FastAPI 官方推荐）。requests 库是同步阻塞的，绝对不能用在异步函数里。

In [ ]:
import httpx
import asyncio
import time

async def fetch_url(client, url):
    print(f"开始请求: {url}")
    start = time.time()
    # 异步发起 GET 请求（相比 requests，这里不会阻塞事件循环）
    response = await client.get(url)
    elapsed = time.time() - start
    print(f"完成请求: {url}, 耗时 {elapsed:.2f}s")
    return response.status_code, len(response.text)

async def main():
    urls = [
        "https://httpbin.org/delay/1",  # 故意延迟 1 秒
        "https://httpbin.org/delay/2",
        "https://httpbin.org/get",
        "https://httpbin.org/status/200",
        "https://httpbin.org/anything",
    ]

    # 使用 async with 管理连接池（节省资源）
    async with httpx.AsyncClient(timeout=10.0) as client:
        tasks = [fetch_url(client, url) for url in urls]
        results = await asyncio.gather(*tasks)

    for url, (status, length) in zip(urls, results):
        print(f"{url} -> 状态码: {status}, 长度: {length}")

# 总耗时约 2 秒（取决于最慢的那个 delay/2），而不是 1+2+... 的总和
asyncio.run(main())

## 5. 实战进阶：带超时控制 + 结果异常处理（大模型必备）

* 在实际调 API 时，必须处理单个请求超时、部分请求失败的情况。这时用 asyncio.wait 或 asyncio.gather(return_exceptions=True)。

In [ ]:
async def safe_fetch(client, url, timeout=3):
    try:
        # 设置每个请求独立超时
        resp = await client.get(url, timeout=timeout)
        return {"url": url, "status": resp.status_code, "data": resp.text[:50]}
    except httpx.TimeoutException:
        return {"url": url, "error": "超时"}
    except Exception as e:
        return {"url": url, "error": str(e)}

async def main_with_errors():
    urls = [
        "https://httpbin.org/delay/1",   # 正常
        "https://httpbin.org/delay/5",   # 会超时（我们设 timeout=2）
        "https://httpbin.org/status/404", # 返回 404，但不算异常，会正常捕获
    ]
    
    async with httpx.AsyncClient() as client:
        tasks = [safe_fetch(client, url, timeout=2) for url in urls]
        results = await asyncio.gather(*tasks)
        asyncio.Semaphore

    for res in results:
        print(res)

asyncio.run(main_with_errors())

## 6. 针对大模型的特殊场景：第一个结果优先（asyncio.wait）

* 如果你同时向多个大模型提问，想要哪个先返回就用哪个（节省时间）

In [ ]:
async def call_llm(model, prompt):
    await asyncio.sleep(1)  # 模拟网络
    return f"{model} 回答了: {prompt}"

async def main_first_completed():
    tasks = [
        asyncio.create_task(call_llm("GPT-4", "你好")),
        asyncio.create_task(call_llm("Claude", "你好")),
        asyncio.create_task(call_llm("Gemini", "你好")),
    ]
    # 等待任意一个任务完成就返回（其余未完成的任务会被自动取消）
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
    # 取消未完成的任务（避免资源浪费）
    for p in pending:
        p.cancel()
    
    # 获取最先完成的结果
    first_result = done.pop().result()
    print(f"最先返回的是: {first_result}")

asyncio.run(main_first_completed())

### 常见陷阱 & 最佳实践总结

| 问题                       | 错误写法                                             | 正确写法                                                     |
| :------------------------- | :--------------------------------------------------- | :----------------------------------------------------------- |
| **调用异步函数**           | `call_llm("你好")`（返回协程对象，没执行）           | `await call_llm("你好")` 或 `asyncio.create_task(...)`       |
| **在 async 函数里做 HTTP** | `import requests; requests.get(url)`（阻塞事件循环） | `async with httpx.AsyncClient() as client; await client.get(url)` |
| **使用 `time.sleep`**      | 在 async 函数中调用 `time.sleep(1)`                  | 用 `await asyncio.sleep(1)`                                  |
| **运行入口**               | `main()`（直接调用会报错）                           | `asyncio.run(main())`                                        |
| **异常处理**               | 用 `gather` 不设置参数，一个异常导致全部崩溃         | 用 `asyncio.gather(*tasks, return_exceptions=True)` 或单独 `try/except` |

------

### 一句话总结

异步编程的终极奥义是：**让 CPU 在等待网络 IO 时，抽空去干别的活**。记住三个核心：**`await` 挂起、`gather` 并发、用 `httpx` 替换 `requests`**。在大模型应用中，正确使用 `asyncio` 能把响应速度提升 3~5 倍，这是必须掌握的核心技能。 😎